# Final Recommender Demo

이 노트북은 `final/engine.py`의 cold/warm 추천 API를 직접 호출해보는 간단한 테스트용 노트북입니다.

공개 함수의 출력은 UI 인수인계 계약에 맞춰 `list[int]`, 즉 top-k `app_id` 목록입니다. 점수와 rank를 보고 싶으면 detail 함수를 사용합니다.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "final":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from final import FinalRecommendationEngine

engine = FinalRecommendationEngine()
print("catalog size:", len(engine.app_ids))
print("known warm users:", len(engine.user_to_idx))

catalog size: 50872
known warm users: 49742


## 1. Cold 추천

입력은 `preferred_genres`, `liked_game_ids`, `interest_weight`, `k`입니다. 출력은 top-k `app_id` 목록입니다.

In [6]:
cold_ids = engine.recommend_cold(
    preferred_genres=["RPG", "Open World"],
    liked_game_ids=[292030],  # The Witcher 3 and Cyberpunk 2077
    interest_weight=0.7,
    k=5,
)
cold_ids

[1091500, 379720, 620, 242760, 812140]

detail 함수는 `rank`, `score`, `content_score_z`, `baseline_score_z`를 함께 보여줍니다.

In [7]:
cold_detail = engine.recommend_cold_detail(
    preferred_genres=["RPG", "Open World"],
    liked_game_ids=[292030],
    interest_weight=0.7,
    k=5,
)
cold_detail

,rank,app_id,score,content_score_z,baseline_score_z,baseline,interest_weight,method
0,1,1091500,3.192887,1.950037,6.092869,train_popularity,0.7,cold_intent_dot_product
1,2,379720,3.186407,1.971041,6.022260,train_popularity,0.7,cold_intent_dot_product
2,3,620,3.058776,1.710570,6.204591,train_popularity,0.7,cold_intent_dot_product
3,4,242760,3.023134,1.705383,6.097887,train_popularity,0.7,cold_intent_dot_product
4,5,812140,3.006680,1.982947,5.395390,train_popularity,0.7,cold_intent_dot_product


catalog metadata와 붙여서 title까지 확인할 수 있습니다.

In [8]:
cold_with_title = cold_detail.merge(
    engine.catalog[["app_id", "title"]],
    on="app_id",
    how="left",
)
cold_with_title[["rank", "app_id", "title", "score", "content_score_z", "baseline_score_z"]]

,rank,app_id,title,score,content_score_z,baseline_score_z
0,1,1091500,Cyberpunk 2077,3.192887,1.950037,6.092869
1,2,379720,DOOM,3.186407,1.971041,6.022260
2,3,620,Portal 2,3.058776,1.710570,6.204591
3,4,242760,The Forest,3.023134,1.705383,6.097887
4,5,812140,Assassin's Creed® Odyssey,3.006680,1.982947,5.395390


## 2. Warm 추천

입력은 `user_id`, `interest_weight`, `k`입니다. 테스트 가능한 user_id 하나를 checkpoint에서 가져옵니다.

In [9]:
sample_user_id = next(iter(engine.user_to_idx))
sample_user_id

13

In [11]:
warm_ids = engine.recommend_warm(
    user_id=sample_user_id,
    interest_weight=0.6,
    k=5,
)
warm_ids

[359550, 304930, 374320, 444090, 252490]

In [12]:
warm_detail = engine.recommend_warm_detail(
    user_id=sample_user_id,
    interest_weight=0.6,
    k=10,
)
warm_with_title = warm_detail.merge(
    engine.catalog[["app_id", "title"]],
    on="app_id",
    how="left",
)
warm_with_title[["user_id", "rank", "app_id", "title", "score", "content_score_z", "baseline_score_z"]]

,user_id,rank,app_id,title,score,content_score_z,baseline_score_z
0,13,1,359550,Tom Clancy's Rainbow Six® Siege,4.014504,2.396777,6.441095
1,13,2,304930,Unturned,3.989647,2.402923,6.369734
2,13,3,374320,DARK SOULS™ III,3.919803,2.240421,6.438875
3,13,4,444090,Paladins®,3.831309,2.243176,6.213510
4,13,5,252490,Rust,3.826393,2.369985,6.011005
5,13,6,377160,Fallout 4,3.789725,2.191921,6.186432
6,13,7,218620,PAYDAY 2,3.772535,2.317894,5.954497
7,13,8,292030,The Witcher® 3: Wild Hunt,3.747927,2.164494,6.123076
8,13,9,105600,Terraria,3.733400,2.230638,5.987543
9,13,10,232090,Killing Floor 2,3.731319,2.396024,5.734260


## 3. interest_weight 변화 확인

`interest_weight`가 낮으면 baseline의 영향이 커지고, 높으면 content dot product의 영향이 커집니다.

In [13]:
for weight in [0.0, 0.5, 1.0]:
    ids = engine.recommend_cold(
        preferred_genres=["RPG", "Open World"],
        liked_game_ids=[292030],
        interest_weight=weight,
        k=5,
    )
    print(f"cold interest_weight={weight}:", ids)

cold interest_weight=0.0: [440, 374320, 550, 377160, 620]
cold interest_weight=0.5: [1091500, 379720, 374320, 620, 242760]
cold interest_weight=1.0: [844850, 1252830, 1337010, 1239080, 955050]


c:\Users\User\26_2_Contest_final\final\engine.py:278: RuntimeWarning: invalid value encountered in multiply
  final = alpha * content_z + (1.0 - alpha) * popularity_z
c:\Users\User\26_2_Contest_final\final\engine.py:278: RuntimeWarning: invalid value encountered in multiply
  final = alpha * content_z + (1.0 - alpha) * popularity_z


In [14]:
for weight in [0.0, 0.6, 1.0]:
    ids = engine.recommend_warm(
        user_id=sample_user_id,
        interest_weight=weight,
        k=5,
    )
    print(f"warm interest_weight={weight}:", ids)

warm interest_weight=0.0: [440, 359550, 374320, 304930, 550]
warm interest_weight=0.6: [359550, 304930, 374320, 444090, 252490]
warm interest_weight=1.0: [213610, 976310, 389730, 1085660, 346010]


c:\Users\User\26_2_Contest_final\final\engine.py:349: RuntimeWarning: invalid value encountered in multiply
  final = alpha * content_z + (1.0 - alpha) * mf_z
c:\Users\User\26_2_Contest_final\final\engine.py:349: RuntimeWarning: invalid value encountered in multiply
  final = alpha * content_z + (1.0 - alpha) * mf_z


## 4. 성능 근거 파일 확인

`final` 폴더의 코드는 새 학습을 하지 않는 inference wrapper입니다. 성능 평가는 upstream artifact에서 수행되었습니다.

In [15]:
import pandas as pd
import json

history_metrics = pd.read_csv(REPO_ROOT / "history_user_tower/results_seed_42/evaluation_metrics.csv")
history_metrics[["model", "evaluation", "recall@10", "ndcg@10", "catalog_coverage@10"]]

,model,evaluation,recall@10,ndcg@10,catalog_coverage@10
0,simple_mean,sampled_1_positive_99_negative,0.768014,0.487032,0.161012
1,hours_weighted_mean,sampled_1_positive_99_negative,0.780925,0.503587,0.154250
2,history_mlp_bpr,sampled_1_positive_99_negative,0.817454,0.535896,0.104910
3,hours_weighted_mean,full_catalog,0.022218,0.011027,0.053015
4,history_mlp_bpr,full_catalog,0.016513,0.007415,0.001533


In [16]:
summary_path = REPO_ROOT / "recommendation_mvp/model_artifacts/multimodal_evaluation_summary_seed42.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
pd.DataFrame(summary["hybrid"]["test_results"])[[
    "model", "recall@10", "ndcg@10", "mrr", "long_tail_recall@10", "catalog_coverage@10"
]]

,model,recall@10,ndcg@10,mrr,long_tail_recall@10,catalog_coverage@10
0,multimodal_only,0.802584,0.518213,0.438291,0.271318,0.159557
1,mf_multimodal_balanced,0.817808,0.561021,0.489525,0.038760,0.113225
2,mf_multimodal_accuracy,0.816607,0.558968,0.487367,0.012403,0.107171
3,mf_only,0.796775,0.538563,0.467079,0.000000,0.097539
